# 06 · Player leaderboards
Qualified (≥500 5v5 min) PM-EPA/60 with bootstrap CIs. Read the CI before the rank.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import polars as pl, numpy as np
import matplotlib.pyplot as plt
from src.config import load_config
cfg = load_config()
proc = cfg.path(cfg.data.processed_dir)
pl.Config.set_tbl_rows(25); pl.Config.set_tbl_width_chars(160)

In [ ]:
lb = pl.read_csv(cfg.path(cfg.data.output_dir) / 'pm_epa_player_leaderboards.csv')
q = lb.filter(pl.col('qualified_ev'))
q.filter(pl.col('season') == q['season'].max()).sort('pm_epa_per60_ev', descending=True).select('player_name','team','position','toi_ev','pm_epa_per60_ev','uncertainty_ci_low_per60','uncertainty_ci_high_per60','pm_direct_per60_ev','pm_pooled_per60_ev','percentile_ev').head(25)

In [ ]:
multi = q.group_by('player_id','player_name','position').agg(pl.col('pm_epa_ev').sum(), pl.col('toi_ev').sum(), pl.len().alias('seasons')).filter(pl.col('seasons') >= 3).with_columns((pl.col('pm_epa_ev') * 60 / pl.col('toi_ev')).alias('per60'))
multi.sort('per60', descending=True).head(25)

In [ ]:
ts = pl.read_csv(cfg.path(cfg.data.output_dir) / 'pm_epa_team_season.csv')
ts.filter(pl.col('season') == ts['season'].max()).sort('pm_epa_attributed', descending=True)